In [ ]:
# BASE CODE

In [ ]:
# ── Generic Imports ────────────
import numpy as np 
from PIL import Image 
import scipy.io as sio
import matplotlib.pyplot as plt

In [ ]:
# ── Constants ──────────────────
RESIZE: int = 256
DATA_PATH: str = "./../data/WillowObject/WILLOW-ObjectClass/"

In [ ]:
# ── Data Class ─────────────────
class NotatedImage:
    # 1. Constructor Method
    def __init__(self, img, kpts) -> None:
        self.img: Image.Image = img 
        self.kpts: np.array = kpts

    # 2. Resize Method
    def resize(self):
        self.kpts[0] *= RESIZE / self.img.size[0]
        self.kpts[1] *= RESIZE / self.img.size[1]
        self.img = self.img.resize((RESIZE, RESIZE), resample=Image.BILINEAR)
        return self

In [ ]:
# ── Path Retrieval Function ──
def getting(cat: str, n: int) -> list[NotatedImage]:
    # Local Imports
    import glob # For searching files
    import os   # To remove file extension

    images: list[str] = glob.glob(f"{DATA_PATH}{cat}/*.png")
    points: list[str] = glob.glob(f"{DATA_PATH}{cat}/*.mat")
    output = []
    set_points = set(points)

    for img_file in images:
        base, _ = os.path.splitext(img_file)
        mat_file = base + ".mat"
        if mat_file in set_points:
            img = Image.open(img_file)
            kpts = np.array(sio.loadmat(mat_file)['pts_coord'])
            ni = NotatedImage(img, kpts)
            output.append(ni)
            if len(output) >= n:
                break

    return output

In [ ]:
# NEW CODE

In [ ]:
class Pair:
    def __init__(self, ni_a, ni_b) -> None:
        self.ni_a = ni_a 
        self.ni_b = ni_b 

In [1]:
def simple_spatial_matching(self):
    # Local Imports
    from scipy.spatial import distance_matrix
    from scipy.optimize import linear_sum_assignment

    # Processing Information
    points_a = np.array(list(zip(self.ni_a.kpts[0], self.ni_a.kpts[1])))
    points_b = np.array(list(zip(self.ni_b.kpts[0], self.ni_b.kpts[1])))

    # Generate Matrixes Containing Euclidean distances
    cost_matrix = distance_matrix(points_a, points_b)
    # Apply the Hungarian Algorithm
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    # Generate Matrix with Zeros
    matching_matrix = np.zeros(cost_matrix.shape, dtype=int)
    # Populate the Matching Matrix
    matching_matrix[row_ind, col_ind] = 1
    
    self.match = matching_matrix
    return self

# We append it to NotatedImage
Pair.add_match = simple_spatial_matching

NameError: name 'Pair' is not defined

In [ ]:
import numpy as np

def calculate_accuracy(self) -> float:
    n = self.match.shape[0]
    ground_truth = np.eye(n) 
    
    # Count how many predicted matches fall exactly on the diagonal
    correct_matches = np.sum((self.match == 1) & (ground_truth == 1))
    
    self.acc = correct_matches / n
    return self

# Append it to the class
Pair.get_accuracy = calculate_accuracy

In [ ]:
def visualize_matching_full(pa: Pair):
    img_a, img_b = pa.ni_a.img, pa.ni_b.img
    kpts_a, kpts_b = pa.ni_a.kpts, pa.ni_b.kpts
    
    # 1. Create composite canvas
    width = max(img_a.size[0], img_b.size[0])
    height = max(img_a.size[1], img_b.size[1])
    composite = Image.new('RGB', (width * 2, height))
    composite.paste(img_a, (0, 0))
    composite.paste(img_b, (width, 0)) 
    
    plt.figure(figsize=(12, 6))
    plt.imshow(composite)
    plt.axis('off')
    
    # 2. Draw Delaunay Structure
    for i, j in pa.ni_a.edges:
        plt.plot([kpts_a[0, i], kpts_a[0, j]], [kpts_a[1, i], kpts_a[1, j]], 'y-', alpha=0.5, lw=1)
        
    for i, j in pa.ni_b.edges:
        plt.plot([kpts_b[0, i] + width, kpts_b[0, j] + width], [kpts_b[1, i], kpts_b[1, j]], 'y-', alpha=0.5, lw=1)
        
    # 3. Matching Lines
    rows, cols = np.where(pa.match == 1)
    for i, j in zip(rows, cols):
        plt.plot([kpts_a[0, i], kpts_b[0, j] + width], [kpts_a[1, i], kpts_b[1, j]], 'g-', lw=1.5, alpha=0.8)
        
    # 4. Keypoints
    plt.scatter(kpts_a[0], kpts_a[1], c='w', edgecolors='k', s=40, zorder=5)
    plt.scatter(kpts_b[0] + width, kpts_b[1], c='w', edgecolors='k', s=40, zorder=5)
    
    plt.title("Graph Matching Results")
    
    # Save and close
    plt.savefig('matching_result.jpg', bbox_inches='tight', dpi=300)
    plt.close()

In [ ]:
# ── Orchestrator Function ───
def show(cat: str, n: int = 4, k: int = 0):
    # Local Import 
    import math 
    imgs = getting(cat, n)                # Gets
    imgs = [img.resize() for img in imgs] # Resizes
    imgs = [img.add_delaunay() for img in imgs]
    
    n = math.floor(len(imgs) / 2)
    head = imgs[:n]
    tail = imgs[n:2*n]
    
    couples = [Pair(ni_a, ni_b) for ni_a, ni_b in zip(head, tail)]
    for p in couples:
        p.add_match()
        p.get_accuracy()
    
    visualize(imgs)                       # And plots